[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nlavadodatasci/atd-megaf-fiabilidade/blob/main/T7_componentes_criticos/T7_componentes_criticos.ipynb)

# T7 — Componentes críticos e alocação de redundância

**ATD — MEGAF 2026/27 · Atividade de Fiabilidade · Tema T7**

Com um orçamento limitado, onde compensa investir? Este *notebook* mostra, num sistema **inventado** de três blocos, como medir a importância de cada componente e como comparar alternativas de investimento.

**Como usar:** execute as células por ordem (Shift+Enter). As primeiras secções estão resolvidas, com dados da aula ou com exemplos inventados, para mostrar o método. A última secção, **A sua vez**, lê os dados da instalação e deixa pistas: o trabalho do grupo começa aí. O capítulo deste tema no documento de apoio explica cada passo.

## 1. Um sistema inventado e a sua função de fiabilidade

Um bloco A em série com dois blocos, B e C, em paralelo:

```
              +-- B --+
 --- A ---o---|       |---o---
              +-- C --+
```

O primeiro passo é escrever **uma função** que recebe as fiabilidades dos blocos e devolve a do sistema. Com ela, qualquer pergunta do tipo "e se...?" passa a ser uma chamada à função com outros valores.

In [1]:
import os, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

REPO = "https://raw.githubusercontent.com/nlavadodatasci/atd-megaf-fiabilidade/main/dados/"

def obter(ficheiro):
    """Vai buscar o ficheiro ao repositório. Se falhar, abre uma janela
    para o escolher no seu computador."""
    if not os.path.exists(ficheiro):
        try:
            urllib.request.urlretrieve(REPO + ficheiro, ficheiro)
        except Exception:
            from google.colab import files
            print("Escolha no seu computador o ficheiro", ficheiro)
            carregados = files.upload()
            if ficheiro not in carregados:               # veio com outro nome: renomeia
                os.rename(list(carregados)[0], ficheiro)
    return ficheiro

def serie(*R):
    return np.prod(R)

def paralelo(*R):
    return 1 - np.prod([1 - r for r in R])

def R_sistema(r):
    # r é um dicionário com a fiabilidade de cada bloco
    return serie(r["A"], paralelo(r["B"], r["C"]))

base = {"A": 0.95, "B": 0.80, "C": 0.70}        # valores inventados
print(f"Fiabilidade do sistema: {R_sistema(base):.4f}")

Fiabilidade do sistema: 0.8930


## 2. Importância de Birnbaum

A importância de Birnbaum de um bloco é a diferença entre a fiabilidade do sistema com esse bloco **perfeito** ($R = 1$) e com esse bloco **avariado** ($R = 0$), mantendo os outros como estão:

$$I_i = R_{\text{sistema}}(R_i = 1) - R_{\text{sistema}}(R_i = 0)$$

Mede quanto o sistema depende do bloco. Um bloco em série com tudo o resto (um **ponto único de falha**) tem sempre importância alta: se falhar, o sistema falha.

In [2]:
def birnbaum(funcao, r):
    resultado = {}
    for bloco in r:
        perfeito = {**r, bloco: 1.0}             # cópia do dicionário com este bloco a 1
        avariado = {**r, bloco: 0.0}
        resultado[bloco] = funcao(perfeito) - funcao(avariado)
    return pd.Series(resultado).sort_values(ascending=False)

birnbaum(R_sistema, base)

A    0.940
B    0.285
C    0.190
dtype: float64

## 3. Comparar alternativas de investimento

A importância diz onde o sistema é mais sensível, mas não diz quanto custa melhorar. Para decidir, é preciso calcular a fiabilidade do sistema **com** cada alternativa e comparar o ganho com o custo.

Acrescentar uma unidade redundante muda a **estrutura** do sistema: o bloco passa a ser um paralelo. Duas alternativas, com custos inventados:

- **Alternativa 1**: um segundo bloco A em paralelo com o primeiro (custo 10);
- **Alternativa 2**: um terceiro bloco, igual a B, no paralelo B–C (custo 4).

In [3]:
r = base
atual = R_sistema(r)
alt1  = serie(paralelo(r["A"], r["A"]), paralelo(r["B"], r["C"]))
alt2  = serie(r["A"], paralelo(r["B"], r["C"], r["B"]))

tab = pd.DataFrame({"R do sistema": [atual, alt1, alt2], "custo": [0, 10, 4]},
                   index=["situação atual", "alt. 1: segundo A", "alt. 2: terceiro bloco B-C"])
tab["ganho"] = tab["R do sistema"] - atual
tab["ganho por unidade de custo"] = tab["ganho"] / tab["custo"].replace(0, np.nan)
tab.round(4)

,R do sistema,custo,ganho,ganho por unidade de custo
situação atual,0.8930,0,0.0000,NaN
alt. 1: segundo A,0.9376,10,0.0446,0.0045
alt. 2: terceiro bloco B-C,0.9386,4,0.0456,0.0114


Neste exemplo, as duas alternativas dão ganhos parecidos, mas uma custa menos de metade da outra. O bloco mais importante segundo Birnbaum não é necessariamente o melhor sítio para gastar o dinheiro: depende do custo e de quanto se consegue de facto melhorar.

## 4. A sua vez

Pistas:

1. Escreva a função `R_sistema` da instalação (a estrutura está descrita na secção "Os dados da instalação" do documento de apoio; é o mesmo trabalho do tema T6, combine com esse grupo). Precisa de $R(2000)$ para cada componente.
2. Calcule a importância de Birnbaum de cada **tipo** de componente e ordene. Há algum ponto único de falha?
3. Leia a folha `custos`. Que combinações de unidades redundantes cabem no orçamento? Faça a lista antes de fazer contas.
4. Para cada alternativa, escreva a estrutura alterada e calcule a fiabilidade do sistema, o ganho e o custo. Compare pelo menos três.
5. A melhor alternativa é a que dá maior fiabilidade, a que dá maior ganho por euro, ou outra? Justifique o critério.
6. Análise de sensibilidade: se o $R(2000)$ de um componente estiver mal estimado (experimente $\pm$ 0,01), a ordenação das alternativas muda?
7. Há alternativas que não sejam comprar unidades redundantes? (Veja o tema T9.)

In [4]:
custos = pd.read_excel(obter("dados_fiabilidade.xlsx"), sheet_name="custos")
custos

,componente,custo_unidade_redundante_eur,custo_substituicao_preventiva_eur,custo_falha_em_servico_eur
0,Interruptor,3000.0,NaN,NaN
1,Transformador,25000.0,NaN,NaN
2,QGBT,8000.0,NaN,NaN
3,Gerador,40000.0,NaN,NaN
4,Controlador,12000.0,2000.0,15000.0
5,NaN,NaN,NaN,NaN
6,Orçamento disponível para melhoria (EUR),50000.0,NaN,NaN
7,Tempo de missão (h),2000.0,NaN,NaN
8,"Nota: valores fictícios, definidos pelo docent...",NaN,NaN,NaN


In [5]:
ensaios = pd.read_excel("dados_fiabilidade.xlsx", sheet_name="ensaios_vida")
missao = 2000

# Fiabilidade de cada componente para a missão (a preencher pelo grupo)
R = {"Interruptor": None, "Transformador": None, "QGBT": None, "Gerador": None, "Controlador": None}

# def R_sistema(r):
#     ramo = serie(r["Interruptor"], r["Transformador"])
#     ...